In [37]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from simulate.interpolant import Interpolant


In [49]:
param = (
    pd.read_csv("data/parameters_2.csv")
    .drop_duplicates()
    .sort_values(["Mode", "State of Charge / 1"], ascending=[False, True])
    .reset_index(drop=True)
).iloc[1::]
keys = [
    "Mode",
    "State of Charge / 1",
    "cost",
    "Open-circuit voltage / V",
    "R0 / Ohm",
]
for i in range(1, 6):
    if f"R{i} / Ohm" in param.columns:
        keys.append(f"R{i} / Ohm")
    if f"Tau{i} / s" in param.columns:
        keys.append(f"Tau{i} / s")
param[keys]

,Mode,State of Charge / 1,cost,Open-circuit voltage / V,R0 / Ohm,R1 / Ohm,Tau1 / s,R2 / Ohm,Tau2 / s
1,Discharge,0.000953,0.005453,2.843882,0.090889,0.367856,100.187965,0.000427,3638.446001
2,Discharge,0.002657,0.005386,2.742336,0.062142,0.261167,28.880536,9.016803,4693.605007
3,Discharge,0.006742,0.004875,2.783973,0.051636,0.244548,33.308134,8.385148,4880.019697
4,Discharge,0.010898,0.004278,2.822695,0.049233,0.228279,41.788820,8.214010,5053.307460
5,Discharge,0.015054,0.003732,2.855840,0.047362,0.216378,55.396166,7.741019,5128.750558
...,...,...,...,...,...,...,...,...,...
478,Charge,0.985447,0.000277,4.150124,0.021073,0.015205,20.181676,0.291878,2092.548515
479,Charge,0.989613,0.000299,4.158301,0.020877,0.014869,23.148084,0.178462,1166.835494
480,Charge,0.993763,0.000309,4.166682,0.023618,0.014615,39.599957,0.259029,1592.814321
481,Charge,0.997693,0.000364,4.176216,0.022705,0.012166,20.633018,0.192897,1057.947730


In [50]:
kinds = {
    "Open-circuit voltage / V": "lut",
    "R0 / Ohm": "spline",
}

options = {
    "Open-circuit voltage / V": {"kind": "cubic"},
    "R0 / Ohm": {"k": 1, "n": 3},
}

for i in range(1, 6):
    if f"R{i} / Ohm" in param.columns:
        kinds[f"R{i} / Ohm"] = "spline"
        options[f"R{i} / Ohm"] = {"k": 1, "n": 3}
    if f"Tau{i} / s" in param.columns:
        kinds[f"Tau{i} / s"] = "spline"
        options[f"Tau{i} / s"] = {"k": 1, "n": 3}

for key in keys[2:]:
    fig = px.scatter(
        param,
        x="State of Charge / 1",
        y=key,
        color="Mode",
    )
    if key in kinds:
        for mode in param["Mode"].unique():
            mask = param["Mode"] == mode
            x = param.loc[mask, "State of Charge / 1"].values
            y = param.loc[mask, key].values
            interpolant = Interpolant(x, y, kind=kinds[key], options=options[key])
            x_interp = np.linspace(0, 1, 1000)
            y_interp = interpolant(x_interp)
            fig.add_trace(
                go.Scatter(
                    x=x_interp,
                    y=y_interp,
                    mode="lines",
                )
            )
    fig.show()